### Notebook to look at individual researchers' corpus  

- Match Domingo's T/C researchers to OA author_ids  
- Extract the full corpus of researchers

In [109]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

import contextlib
from unidecode import unidecode
from nameparser import HumanName

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def normalise_name(in_name: str=None) -> dict:
    # print(f'{in_name = }')
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    if name.middle == "":
        fullname = f'{name.first} {name.last}'
    else:
        fullname = f'{name.first} {name.middle} {name.last}'
    return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}

In [110]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return

    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
    
        with contextlib.suppress(Exception):
            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(str), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
            # self.db.create_function('extract_works', 
            #                             extract_works, 
            #                             return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
            #                         )
        self.db.sql("SHOW ALL TABLES").show()
        return

In [111]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    def extract_citation_time_series(self):
        sql = """
                CREATE OR REPLACE TABLE memory.endogenous_citations AS
                SELECT count(sub.work_id) AS cited_by_count_endogenous,
                        sub.author_id,
                        sub.author_name,
                        sub.publication_year
                FROM
                    (SELECT DISTINCT w.work_id,
                            w.cited_by_count,
                            a.author_id,
                            a.author_name,
                            w.publication_year
                    FROM works w
                    LEFT JOIN (SELECT work_id,
                                unnest(referenced_works) AS cited_id
                                FROM cited
                            ) c
                ON w.work_id = c.cited_id
                LEFT JOIN authorships a
                ON a.work_id = w.work_id
                WHERE a.author_id NOT NULL) sub
                GROUP BY ALL
                ORDER BY publication_year ASC, cited_by_count_endogenous DESC
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.endogenous_citations").show()
        return
    
    def get_hcp(self):
        df = self.db.sql("SELECT * FROM memory.endogenous_citations").df()
        print(f'{df.shape = }\n{df.head()}')
        groups = df.groupby('publication_year').cited_by_count_endogenous.quantile(0.99).to_frame().rename(columns={'cited_by_count_endogenous': 'centile'})
        print(f'{groups.shape = }\n{groups.head(32)}')
        df = df.set_index('publication_year').join(groups)
        print(f'{df.shape = }\n{df.head()}')
        df = df[df.cited_by_count_endogenous >= df.centile].reset_index()
        print(f'{df.shape = }\n{df.head()}')
        groups = df.groupby(['author_id', 'author_name']).publication_year.count().\
            reset_index().rename(columns={'publication_year': 'hcp_count'}).sort_values('hcp_count', ascending=False).reset_index(drop=True)
        names = pd.DataFrame.from_dict([normalise_name(n) for n in groups.author_name])
        print(f'{names.shape = }\n{names.head()}')
        groups = pd.concat([groups, names], axis=1)
        print(f'{groups.shape = }\n{groups.head()}')
        self.db.sql("CREATE OR REPLACE TABLE econ.hcp_count_endogenous AS SELECT * FROM groups")
        self.db.sql("SELECT * FROM econ.hcp_count_endogenous").show()
        return

In [112]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample['Research_Profile'] = [normalise_name(n).get('fullname') for n in sample.Research_Profile]
        sample['first'] = [normalise_name(n).get('first') for n in sample.Research_Profile]
        sample['last'] = [normalise_name(n).get('family') for n in sample.Research_Profile]
        sample['found'] = False
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        print('Duplicate names in Domingo list?')
        print(sample[sample.duplicated(keep=False)].head())
        self.db.sql("CREATE OR REPLACE TABLE econ.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        return
    
    def compare_sample_hca(self):
        self.hca = self.db.sql("SELECT * FROM econ.hcp_count_endogenous").df()
        self.hca = self.hca[self.hca.hcp_count > 3]
        self.sample_test = self.sample[self.sample['Group'] == 'T']
        print(f'{self.hca.shape = }\n{self.hca.head(128)}')
        print(f'{self.sample_test.shape = }\n{self.sample_test.head(128)}')
        for row in self.sample_test.itertuples():
            found = False
            for row1 in self.hca.itertuples():
                if row.Research_Profile == row1.author_name:
                    print(f'MATCH full {row.Index} {row.Research_Profile = }')
                    self.sample.at[row.Index, 'found'] = True
                    found = True
                    continue
        self._compare_sample_hca()
        print(f'{self.sample_test.shape = }\n{self.sample_test.head(32)}')
        print(f'{self.sample_test[self.sample_test.found].shape = }\n{self.sample_test[self.sample_test.found].head(16)}')
        print(f'{self.sample_test[~self.sample_test.found].shape = }\n{self.sample_test[~self.sample_test.found].head(16)}')
        return
    
    def _compare_sample_hca(self):
        for row in self.sample_test.itertuples():
            if row.found:
                continue
            found = False
            for row1 in self.hca.itertuples():
                if row1.hcp_count < 3:
                    continue
                if row.last.lower() == row1.family.lower() and row.first[0] == row1.first[0]:
                    print(f'match LAST {row.Index} {row.Research_Profile = }')
                    self.sample_test.at[row.Index, 'found'] = True
                    found = True
                    continue
        return                          
         
    
    def match_sample_fullname(self):
        sql = """ 
            SELECT Research_Profile,
                    count(Research_Profile) AS match_count_domingo,
                    count(DISTINCT author_id) AS match_count_openalex,
                    author_id,
                    author_name,
                    found = True
                    continue
            if not found:
     
                    "Group", 
                    CIT,
                    PUB,
                    HCP,
                    h_index,
                    s.first,
                    s.last,
                    works_count,
                    array_to_string(list(DISTINCT country_code), ' ') AS country_codes,
                    array_to_string(list(DISTINCT institution_name), ' ') AS institution_names,
                FROM econ.domingo_sample s
                    LEFT JOIN econ.authors a
                        ON lower(a.author_name) = lower(s.Research_Profile) 
                            -- OR list_contains(str_split(a.author_name, ' '), last) = true
                    --WHERE a.author_id NOT NULL
                    GROUP BY ALL
                    ORDER BY s.last ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        df.to_csv('../DATA/domingo_sample_full_name_match.csv')
        df = df.drop_duplicates() #subset='last')
        print(f'>>fullname match {df.shape = }\n{df.head()}')
        # self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return
    
    def match_sample_lastname(self):
        sql = """ 
            SELECT Research_Profile,
                    author_id,
                    author_name,
                    "Group", 
                    CIT,
                    PUB,
                    HCP,
                    h_index,
                    s.first,
                    s.last,
                    works_count,            print(f'{sample.shape = }\n{sample.head()}')
                    array_to_string(list(country_code), ' ') AS country_codes,
                    array_to_string(list(institution_name), ' ') AS institution_names,
                FROM econ.domingo_sample s
                    LEFT JOIN econ.authors a
                        ON list_contains(str_split(a.author_name, ' '), s.last) = true 
                            AND author_name[1] = Research_Profile[1]
                    WHERE a.author_id NOT NULL
                    GROUP BY ALL
                    ORDER BY s.last ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        df.to_csv('../DATA/domingo_sample_last_name_match.csv')
        df = df.drop_duplicates()  #subset='last')
        print(f'last name match{df.shape = }\n{df.head()}')
        # self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return


In [113]:
def main():

    # cetl = CorpusETL()
    # cetl.extract_citation_time_series() #(author_id='https://openalex.org/A5101600363')
    # cetl.get_hcp()

    mds = MatchDomingoSample()
    mds.extract_sample()
    mds.compare_sample_hca()
    # mds.match_sample_fullname()
    # mds.match_sample_lastname()


        
    return

In [114]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬───────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────